In [21]:
import numpy as np
from sklearn.utils import resample
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

### What is Bootstrapping?

In plain English: You sample **WITH replacement** from your dataset to create many "fake" datasets. This lets you estimate how stable your model is.


**Original Dataset:** `[1, 2, 3, 4, 5]`

Bootstrap Sample 1: [1, 2, 2, 4, 5]

Bootstrap Sample 2: [1, 1, 3, 4, 4]

Bootstrap Sample 3: [2, 3, 3, 5, 5]

Bootstrap Sample 4: [1, 3, 4, 4, 5]

- Each sample is the **same size** as the original, but some values are repeated and some are missing!


- **Bootstrapping** = Create **fake** datasets by sampling with replacement.

- It's like **cloning your data** to see how stable your model really is! 

In [4]:
# original data
data = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
print(f'original data: {data}')

# 5 bootstrap samples
for i in range(5):
    bootstrap_sample = resample(data, n_samples=len(data), replace=True)
    print(f"Sample {i+1}: {bootstrap_sample}")

original data: [ 1  2  3  4  5  6  7  8  9 10]
Sample 1: [ 3  5  1  4 10  7  2  2 10 10]
Sample 2: [ 3  1  8  1  9  8  5 10  7  5]
Sample 3: [4 7 2 9 2 7 3 9 7 2]
Sample 4: [ 6  3 10  5 10  8  5  3  5  3]
Sample 5: [ 1  2  3  7  4  2  8  1  2 10]


### Bootstrapping vs Cross-Validation

| Aspect | Bootstrapping | Cross-Validation |
|--------|---------------|------------------|
| **Sampling** | With replacement | Without replacement |
| **Purpose** | Estimate stability, confidence intervals | Estimate generalization performance |
| **Use in** | Bagging, Random Forest | Model selection, hyperparameter tuning |
| **Data used** | Same size as original | Smaller (train on k-1/k) |

In [13]:
X, y = make_regression(n_samples=200, n_features=5, noise=0.1, random_state=42)
model = RandomForestRegressor(random_state=42)

In [14]:
def bootstrap_metric(model, X, y, n_bootstrap=1000):
    """
    Estimate uncertainty using bootstrapping.
    """
    n = len(X)
    metrics = []
    
    for _ in range(n_bootstrap):
        # Resample with replacement
        X_boot, y_boot = resample(X, y, n_samples=n, replace=True)
        
        # Train on bootstrap sample
        model.fit(X_boot, y_boot)
        
        # Predict on out-of-bag (OOB) samples
        # OOB = samples NOT in the bootstrap sample
        oob_mask = ~np.isin(np.arange(n), np.unique(np.random.choice(n, n, replace=True)))
        if np.sum(oob_mask) > 0:
            X_oob = X[oob_mask]
            y_oob = y[oob_mask]
            y_pred = model.predict(X_oob)
            metrics.append(np.mean((y_oob - y_pred)**2))
    
    return np.array(metrics)

# Use bootstrap
bootstrap_scores = bootstrap_metric(model, X, y, n_bootstrap=100)

print(f"Bootstrap MSE: {bootstrap_scores.mean():.4f} ± {bootstrap_scores.std():.4f}")
print(f"95% CI: [{np.percentile(bootstrap_scores, 2.5):.4f}, {np.percentile(bootstrap_scores, 97.5):.4f}]")

Bootstrap MSE: 379.5967 ± 154.4909
95% CI: [178.7728, 766.2751]


### Why Use Out-of-Bag (OOB) Samples in Bootstrapping?

We **could** just resample and train. But OOB gives us **unbiased evaluation** without needing a separate test set!


The Problem: Evaluating on Training Data

```python
# ❌ This gives overly optimistic (biased) scores!
def bootstrap_biased(model, X, y, n_bootstrap=100):
    n = len(X)
    metrics = []
    
    for _ in range(n_bootstrap):
        # Train on bootstrap sample
        model.fit(X_boot, y_boot)
        
        # ❌ Evaluate on the SAME data we trained on!
        pred = model.predict(X_boot)
        metrics.append(np.mean((y_boot - pred)**2))
    
    return metrics

In [15]:
# ❌ BIASED: Evaluate on training data
def biased_bootstrap(model, X, y, n_bootstrap=50):
    n = len(X)
    metrics = []
    for _ in range(n_bootstrap):
        X_boot, y_boot = resample(X, y, n_samples=n, replace=True)
        model.fit(X_boot, y_boot)
        pred = model.predict(X_boot)  # ← SAME data!
        metrics.append(np.mean((y_boot - pred)**2))
    return np.array(metrics)

# ✅ UNBIASED: Evaluate on OOB
def unbiased_bootstrap(model, X, y, n_bootstrap=50):
    n = len(X)
    metrics = []
    for _ in range(n_bootstrap):
        X_boot, y_boot = resample(X, y, n_samples=n, replace=True)
        model.fit(X_boot, y_boot)
        
        # Find OOB indices
        all_indices = np.arange(n)
        boot_indices = np.unique(np.random.choice(n, n, replace=True))
        oob_mask = ~np.isin(all_indices, boot_indices)
        
        if np.sum(oob_mask) > 0:
            X_oob = X[oob_mask]
            y_oob = y[oob_mask]
            pred = model.predict(X_oob)  # ← UNSEEN data!
            metrics.append(np.mean((y_oob - pred)**2))
    return np.array(metrics)

# Compare
biased_scores = biased_bootstrap(model, X, y)
unbiased_scores = unbiased_bootstrap(model, X, y)

print(f"❌ Biased (train on same data): {biased_scores.mean():.4f} ± {biased_scores.std():.4f}")
print(f"✅ Unbiased (OOB evaluation): {unbiased_scores.mean():.4f} ± {unbiased_scores.std():.4f}")

❌ Biased (train on same data): 59.8987 ± 10.9526
✅ Unbiased (OOB evaluation): 392.7203 ± 144.3416


### Bootstrapping with Train-Test Split (No OOB Needed)

You can split data first, then bootstrap **only on the training set**, and evaluate on the fixed test set. This removes the need for OOB.


In [18]:
# Step 1: Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

# Step 2: Bootstrap on training set only
def bootstrap_with_test(model, X_train, y_train, X_test, y_test, n_bootstrap=100):
    """
    Bootstrap on training set, evaluate on fixed test set.
    No OOB needed!
    """
    n = len(X_train)
    test_scores = []
    
    for _ in range(n_bootstrap):
        # Resample with replacement from TRAINING set
        X_boot, y_boot = resample(X_train, y_train, n_samples=n, replace=True)
        
        # Train on bootstrap sample
        model.fit(X_boot, y_boot)
        
        # Evaluate on FIXED test set (unseen data)
        pred = model.predict(X_test)
        test_scores.append(np.mean((y_test - pred)**2))
    
    return np.array(test_scores)

# Run bootstrap
scores = bootstrap_with_test(model, X_train, y_train, X_test, y_test, n_bootstrap=100)

print(f"\nTest MSE: {scores.mean():.4f} ± {scores.std():.4f}")
print(f"95% CI: [{np.percentile(scores, 2.5):.4f}, {np.percentile(scores, 97.5):.4f}]")

Train size: 160, Test size: 40

Test MSE: 1107.9641 ± 271.1886
95% CI: [750.4189, 1650.9373]


### Why Are the Results So Different?

This is **expected behavior**! Each method answers a **different question**.


What Each Method Actually Measures

| Method | What It Measures | Question It Answers |
|--------|------------------|---------------------|
| **Biased (train on same data)** | Training error on bootstrap samples | "How well does the model memorize the training data?" |
| **OOB Evaluation** | Error on unseen (OOB) data within bootstrap | "How well does the model generalize to new data?" |
| **Train-Test + Bootstrap** | Error on a fixed test set | "How well does the model perform on a specific held-out set?" |


Why They're So Different

**1. Biased (59.9):** Model sees the exact same data it trained on → Overly optimistic! ❌

**2. OOB (392.7):** Model evaluates on data it hasn't seen (37% OOB) → More realistic ✅

**3. Train-Test (1107.9):** Model evaluates on a **completely held-out test set** (different distribution) → Most conservative ✅



In [19]:
from sklearn.model_selection import cross_val_score

# 5-fold cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
mse_scores = -cv_scores

print(f"CV MSE: {mse_scores.mean():.4f} ± {mse_scores.std():.4f}")
print(f"Individual folds: {mse_scores}")

CV MSE: 757.0563 ± 251.2104
Individual folds: [ 665.6151964  1050.89644526 1061.22482406  505.50113811  502.04377365]


#### What These Numbers Tell You

| Method | MSE | Interpretation |
|--------|-----|----------------|
| **Biased (Train on Same)** | 59.90 | ❌ Overly optimistic (cheating!) |
| **OOB Evaluation** | 392.72 | ✅ Reasonable estimate |
| **Train-Test Bootstrap** | 1107.96 | ⚠️ Unlucky test set |
| **Cross-Validation** | **757.06** | ✅ **Most reliable estimate** |



### When is Bootstrapping the Right Method?

Bootstrapping **is** a valid method! It just serves a **different purpose** than cross-validation.


The Key Difference: Purpose

| Method | Primary Purpose | What It Tells You |
|--------|-----------------|-------------------|
| **Cross-Validation** | Estimate **generalization performance** | "How well does my model perform on new data?" |
| **Bootstrapping** | Estimate **uncertainty/stability** | "How confident am I about my model's performance?" |


When to Use Bootstrapping

| Scenario | Why Bootstrapping is Better |
|----------|----------------------------|
| **Small datasets** | You can't afford to waste data on test sets (CV uses less data) |
| **Estimating confidence intervals** | CV gives one number; bootstrapping gives a distribution |
| **Model stability** | See how much your model varies with different data |
| **Ensemble methods** | Bagging and Random Forest are built on bootstrapping |
| **When you don't have enough data for CV** | Works with very small datasets |

When to Use Cross-Validation

| Scenario | Why CV is Better |
|----------|------------------|
| **Model selection** | Comparing models (CV is more standard) |
| **Hyperparameter tuning** | Grid search uses CV |
| **Large datasets** | Enough data to split into folds |
| **Standard benchmark** | CV is the industry standard |


Bootstrapping is NOT for Performance Estimation!

**Bootstrapping is great for:**
- ✅ Confidence intervals
- ✅ Model stability
- ✅ Uncertainty quantification
- ✅ Bagging/Random Forest

**Bootstrapping is NOT great for:**
- ❌ Estimating model performance (CV is better)
- ❌ Comparing models (CV is the standard)
- ❌ Hyperparameter tuning (CV is the standard)

